# Persona-Based Anime Chatbot — Mai Sakurajima

A Transformer-based persona chatbot built on **DialoGPT-medium**, running entirely free on Google Colab (CPU or T4 GPU).

**Architecture**
- `config/config.json` — persona description and decoding settings
- `src/persona.py` — injects persona prefix into every turn
- `src/memory.py` — rolling summarization of the last *N* turns
- `src/model.py` — DialoGPT wrapper that composes persona + memory + dialogue
- `src/evaluation.py` — BLEU, perplexity, persona-vs-baseline comparison
- `data/anime_dialogues.txt` — 30 in-character reference dialogues

Reproducibility: `torch.manual_seed(42)`.

**To run in Google Colab:** just click `Runtime → Run all`. The first cell auto-clones the repo and installs dependencies.

## Section 1 — Setup & Installation

Auto-detects Colab, clones the repo if needed, removes the pre-installed `torchvision` that conflicts with `transformers`, and installs project dependencies.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/OrangGilar/Testing-Kanojo.git'
REPO_DIR = 'Testing-Kanojo'

# Clone the repo if running on Colab and project files aren't already present.
if IN_COLAB and not os.path.exists('src'):
    if not os.path.exists(REPO_DIR):
        !git clone -q $REPO_URL
    os.chdir(REPO_DIR)
    print('Working directory:', os.getcwd())

# Fix Colab's torchvision circular-import that breaks `from transformers import GPT2LMHeadModel`.
# We don't need torchvision/torchaudio for this project — removing them is the cleanest fix.
if IN_COLAB:
    !pip uninstall -y -q torchvision torchaudio

# Install project dependencies (use Colab's pre-installed torch as-is to avoid version mismatches).
# bitsandbytes + accelerate are needed for 4-bit quantization of the 7B model on Colab T4.
%pip install -q --upgrade transformers nltk sentencepiece bitsandbytes accelerate

import nltk
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass

print('Setup complete.')

In [ ]:
import os, sys
if 'src' not in os.listdir('.'):
    raise RuntimeError("Run this notebook from the project root (the folder containing src/, config/, data/).")
sys.path.insert(0, os.getcwd())

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Section 2 — Persona Configuration

Loads **Mai Sakurajima** from `config/config.json` and shows the persona prefix that will be injected into every prompt.

In [ ]:
from src.persona import Persona

persona = Persona.from_config('config/config.json')
print(f"Loaded persona: {persona.name}, age {persona.age}")
print()
print('--- Persona prefix preview ---')
print(persona.build_prefix())

In [ ]:
# Build the chatbot. First run downloads DialoGPT-medium (~350MB).
from src.model import PersonaChatbot

chatbot = PersonaChatbot.from_config('config/config.json')
print(f"Model loaded on: {chatbot.device}")
print(f"Greeting: {persona.greeting}")

## Section 3 — Talk to Mai

**How to chat:**
1. Run the cell below (▶ button or Shift+Enter).
2. Mai will greet you first; type your message in the input box that appears and press Enter.
3. Keep typing to continue the conversation — the transcript builds up as you go.
4. Type **`quit`**, **`exit`**, or **`bye`** to end the conversation.
5. Run the **reset** cell below to wipe Mai's memory and start a fresh transcript.

In [ ]:
# Type your messages in the input box that appears below.
# Type "quit", "exit", or "bye" to end the conversation.

if 'transcript' not in globals():
    transcript = [(chatbot.persona.name, chatbot.persona.greeting)]
    print(f"{chatbot.persona.name}: {chatbot.persona.greeting}\n")

while True:
    try:
        my_message = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nConversation interrupted.")
        break

    if not my_message:
        continue
    if my_message.lower() in ("quit", "exit", "bye"):
        print(f"{chatbot.persona.name}: ...Take care.")
        break

    reply = chatbot.generate(my_message)
    transcript.append(('You', my_message))
    transcript.append((chatbot.persona.name, reply))
    print(f"{chatbot.persona.name}: {reply}\n")

In [ ]:
# Run this cell to clear Mai's memory and the on-screen chat transcript.
chatbot.reset_conversation()
transcript = [(chatbot.persona.name, chatbot.persona.greeting)]
print('Memory cleared. Mai forgot the previous conversation.')

## Section 4 — Memory Demo

Sends a fixed sequence of turns and shows how the rolling summary is produced once the memory window fills up.

In [ ]:
chatbot.reset_conversation()
demo_turns = [
    "Hey Mai, did you wait long?",
    "Sorry, my last class ran over.",
    "Want to grab some pudding from the convenience store?",
    "I'll pay this time, I promise.",
    "By the way, I saw you on TV last night.",
    "You looked really cool in that scene.",
    "Do you ever get nervous before filming?",
    "I always thought you'd be totally fearless.",
    "Anyway, what should we do this weekend?",
    "Maybe a movie? Your pick.",
]

for i, turn in enumerate(demo_turns, 1):
    reply = chatbot.generate(turn)
    print(f"[{i:2d}] You: {turn}")
    print(f"     {chatbot.persona.name}: {reply}")

print('\n--- Rolling summary after the window filled ---')
print(chatbot.memory.summary or '(none yet)')

## Section 5 — Evaluation Results

Compares **persona-conditioned** vs. **baseline (no persona)** generation against the reference replies in `data/anime_dialogues.txt`, then reports BLEU and a held-out perplexity figure.

In [ ]:
from src.evaluation import (
    load_dialogues,
    compute_perplexity,
    compare_persona_vs_baseline,
    format_comparison_table,
)

pairs = load_dialogues('data/anime_dialogues.txt')
print(f"Loaded {len(pairs)} reference dialogue pairs.")

In [ ]:
# Subsample for speed on CPU. Bump up to len(pairs) if you have a GPU.
import random
random.seed(42)
subset = pairs[:10]

result = compare_persona_vs_baseline(chatbot, subset, smoothing=True, verbose=False)
print(format_comparison_table(result, max_rows=10))

In [ ]:
# Perplexity on the dialogue corpus (lower = better-aligned with the model's prior).
corpus_text = '\n'.join(f"User: {p.user}\n{chatbot.persona.name}: {p.reference}" for p in pairs)
ppl = compute_perplexity(corpus_text, chatbot.model, chatbot.tokenizer, device=chatbot.device, stride=512)
print(f"Perplexity on anime_dialogues.txt: {ppl:.2f}")

### Takeaways

- The persona-conditioned prompt nudges DialoGPT toward Mai's tone, but BLEU against a single reference reply is a noisy signal — read the side-by-side table qualitatively.
- Perplexity rises on stylized in-character dialogue because vanilla DialoGPT was trained on generic Reddit chatter; this is expected and motivates persona prompting (or future fine-tuning).
- Memory summarization keeps prompt length bounded as conversations grow.